In [ ]:
!unzip multilabel-classification-dataset.zip -d /content/dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Path where you unzipped dataset
dataset_path = "/content/dataset"

# Walk through files and calculate total size
total_size = 0
for dirpath, dirnames, filenames in os.walk(dataset_path):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        total_size += os.path.getsize(fp)

# Convert to MB
print(f" Dataset size: {total_size / (1024*1024):.2f} MB")


In [ ]:
import pandas as pd

# Load your CSV (adjust if name is different)
df = pd.read_csv("/content/dataset/train.csv")

# Define the label columns
label_columns = [
    'Computer Science',
    'Physics',
    'Mathematics',
    'Statistics',
    'Quantitative Biology',
    'Quantitative Finance'
]

# Count how many samples belong to each class (sum since multilabel has 0/1 per class)
class_counts = df[label_columns].sum().astype(int)

print(" Class distribution (number of samples per label):")
print(class_counts)


title + Abstract

In [ ]:
X = (df["TITLE"] + " " + df["ABSTRACT"]).tolist()


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

encodings = tokenizer(
    X,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors="pt"
)


In [ ]:
import torch


In [ ]:
label_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']
labels = torch.tensor(df[label_cols].values, dtype=torch.float32)


In [ ]:
input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]
labels = torch.tensor(df[label_cols].values, dtype=torch.float32)


In [ ]:
pip install numpy

In [ ]:
import torch
labels = torch.tensor(df[label_cols].values, dtype=torch.float32)

In [ ]:
from skmultilearn.model_selection import IterativeStratification
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import torch.nn as nn

In [ ]:
!pip install scikit-multilearn

In [ ]:
#  Stratified Split
msss = IterativeStratification(n_splits=2, order=1, sample_distribution_per_fold=[0.8, 0.2])

# Convert labels to a sparse matrix
import scipy.sparse as sp
labels_sparse = sp.lil_matrix(labels.numpy())

train_idx, val_idx = next(msss.split(np.arange(len(df)), labels_sparse))

train_dataset = TensorDataset(
    input_ids[train_idx],
    attention_mask[train_idx],
    labels[train_idx]
)

val_dataset = TensorDataset(
    input_ids[val_idx],
    attention_mask[val_idx],
    labels[val_idx]
)

#  Safe DataLoader setup
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

In [ ]:
#  Model + Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=labels.shape[1]
).to(device)

### Model Training

In [ ]:
#  Optimizer and loss
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

#  To store metrics
history = {
    'train_loss': [], 'val_loss': [],
    'train_accuracy': [], 'val_accuracy': [],
    'train_precision': [], 'val_precision': [],
    'train_recall': [], 'val_recall': [],
    'train_f1': [], 'val_f1': []
}

best_f1 = 0.0
epochs = 10  #  Increased to 10 epochs

for epoch in range(epochs):
    print(f"\n===== Epoch {epoch+1}/{epochs} =====")

    # ----------------- TRAINING -----------------
    model.train()
    total_loss = 0
    all_preds, all_targets = [], []

    train_loop = tqdm(train_loader, desc=f"Training Epoch {epoch+1}", leave=False)
    for batch in train_loop:
        ids, mask, targets = [b.to(device) for b in batch]
        optimizer.zero_grad()
        outputs = model(input_ids=ids, attention_mask=mask).logits
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = torch.sigmoid(outputs).detach().cpu().numpy()
        preds = (preds >= 0.5).astype(int)
        all_preds.extend(preds)
        all_targets.extend(targets.cpu().numpy())

        # Show live loss in tqdm bar
        train_loop.set_postfix(loss=loss.item())

    #  Compute train metrics
    train_loss = total_loss / len(train_loader)
    train_acc = accuracy_score(all_targets, all_preds)
    train_prec = precision_score(all_targets, all_preds, average='micro', zero_division=0)
    train_rec = recall_score(all_targets, all_preds, average='micro', zero_division=0)
    train_f1 = f1_score(all_targets, all_preds, average='micro', zero_division=0)

    # ----------------- VALIDATION -----------------
    model.eval()
    val_loss = 0
    val_preds, val_targets = [], []
    val_loop = tqdm(val_loader, desc=f"Validating Epoch {epoch+1}", leave=False)

    with torch.no_grad():
        for batch in val_loop:
            ids, mask, targets = [b.to(device) for b in batch]
            outputs = model(input_ids=ids, attention_mask=mask).logits
            loss = criterion(outputs, targets)
            val_loss += loss.item()

            preds = torch.sigmoid(outputs).cpu().numpy()
            preds = (preds >= 0.5).astype(int)
            val_preds.extend(preds)
            val_targets.extend(targets.cpu().numpy())

            # Show live val loss in tqdm bar
            val_loop.set_postfix(loss=loss.item())

    #  Compute val metrics
    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_targets, val_preds)
    val_prec = precision_score(val_targets, val_preds, average='micro', zero_division=0)
    val_rec = recall_score(val_targets, val_preds, average='micro', zero_division=0)
    val_f1 = f1_score(val_targets, val_preds, average='micro', zero_division=0)

    #  Save metrics
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    history['train_precision'].append(train_prec)
    history['val_precision'].append(val_prec)
    history['train_recall'].append(train_rec)
    history['val_recall'].append(val_rec)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)

    #  Epoch summary — nicely formatted output
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Prec: {train_prec:.4f} | Val Prec: {val_prec:.4f} | "
          f"Train Rec: {train_rec:.4f} | Val Rec: {val_rec:.4f} | "
          f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

    #  Optional progress print for clarity
    print(f" Metrics after Epoch {epoch+1}: "
          f"Train F1={train_f1:.4f}, Val F1={val_f1:.4f}")

    #  Save model after each epoch
    torch.save(model.state_dict(), f"deberta_epoch{epoch+1}.pth")

    #  Save best model based on Val F1
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "deberta_best_model_10_epoch_updated.pth")
        print(f" Best model updated! (Val F1: {best_f1:.4f})")

#  Save metrics for plotting later
torch.save(history, "train_val_metrics.pth")
print("\n Training complete! Metrics and best model saved.")


**Evaluating with the Best Model**




In [ ]:
from transformers import AutoModelForSequenceClassification
import torch
import numpy as np
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, precision_score, recall_score

# Assuming device and label_cols are already defined from previous cells
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
label_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']

# Load the best model
best_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(label_cols)
).to(device)
best_model.load_state_dict(torch.load("deberta_best_model_10_epoch_updated.pth"))
best_model.eval() # Set model to evaluation mode

# Re-run prediction on the validation set using the best model
val_preds_best_model = []
val_targets_best_model = []

with torch.no_grad():
    for batch in val_loader:
        ids, mask, targets = [b.to(device) for b in batch]
        outputs = best_model(input_ids=ids, attention_mask=mask).logits
        preds = torch.sigmoid(outputs).cpu().numpy()
        preds = (preds >= 0.5).astype(int)
        val_preds_best_model.extend(preds)
        val_targets_best_model.extend(targets.cpu().numpy())

val_preds_best_model = np.array(val_preds_best_model)
val_targets_best_model = np.array(val_targets_best_model)

# Calculate and print metrics for the best model
micro_f1_best = f1_score(val_targets_best_model, val_preds_best_model, average='micro')
macro_f1_best = f1_score(val_targets_best_model, val_preds_best_model, average='macro')
micro_prec_best = precision_score(val_targets_best_model, val_preds_best_model, average='micro')
micro_rec_best = recall_score(val_targets_best_model, val_preds_best_model, average='micro')
ham_loss_best = hamming_loss(val_targets_best_model, val_preds_best_model)
acc_score_best = accuracy_score(val_targets_best_model, val_preds_best_model)

print("\n Model: DeBERTa (TITLE + ABSTRACT) - Best Model Evaluation")
print(f"Accuracy Score: {acc_score_best}")
print(f"Micro F1 Score: {micro_f1_best}")
print(f"Macro F1 Score: {macro_f1_best}")
print(f"Micro Precision: {micro_prec_best}")
print(f"Micro Recall: {micro_rec_best}")
print(f"Hamming Loss: {ham_loss_best}")

# Per-class accuracy for the best model
domain_names = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']
class_accs_best = []
for i, name in enumerate(domain_names):
    correct = (val_preds_best_model[:, i] == val_targets_best_model[:, i]).sum()
    acc = correct / len(val_targets_best_model)
    class_accs_best.append(acc)
    print(f"('{name}', {acc})")

print(f"Average accuracy = {np.mean(class_accs_best)}")

In [ ]:
from transformers import DebertaForSequenceClassification, DebertaTokenizer

# after training is complete
output_dir = "/content/drive/MyDrive/deberta_multilabel_model_Finetuned_10epochs_updated"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(" DeBERTa model saved to:", output_dir)